# Portable Model Intake Workflow


**Purpose.** Exercise one complete model-intake path from creation and physical description through linting, transformation, persistence, reconstruction, and export.


**Lattice model.** A finite open tight-binding chain is represented by portable geometry, one orbital degree per site, explicit basis mappings, hopping interactions, and onsite terms.


**Variables.** $N$ is the site count, $t$ is the hopping amplitude, and $H$ is the reconstructed single-particle Hamiltonian.


The workflow below uses only public package APIs. Temporary files keep the notebook repeatable and leave no repository artifacts behind.


In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np

from quantum_lattice_models import (
    adapter_capability_report,
    create_model_spec,
    describe_model,
    export_hamiltonian_artifact,
    lint_model,
    load_model,
    particle_model_vacancies,
)

In [2]:
model = create_model_spec(
    "tight_binding_chain",
    parameters={"n_sites": 6, "hopping": 1.0, "onsite": 0.2},
)
summary = describe_model(model)
lint = lint_model(model)
capability = adapter_capability_report(model, "graphml")

print("Intake summary")
print(f"  family: {summary.family}")
print(f"  lattice sites: {summary.lattice_sites}")
print(f"  local degrees: {summary.local_degrees}")
print(f"  interactions: {summary.interactions}")
print(f"  lint valid: {lint.valid}")
print(f"  GraphML supported: {capability.supported}")

Intake summary
  family: tight_binding_chain
  lattice sites: 6
  local degrees: 6
  interactions: 11
  lint valid: True
  GraphML supported: True


In [3]:
transformed = particle_model_vacancies(model, [2])
with TemporaryDirectory() as directory:
    root = Path(directory)
    model_path = transformed.save(root / "model.json")
    restored = load_model(model_path)
    result = restored.build_result()
    outputs = export_hamiltonian_artifact(result, root / "model.bundle", artifact="bundle")
    print("Round-trip and export")
    print(f"  transformed sites: {restored.lattice.n_sites}")
    print(f"  matrix shape: {result.matrix.shape}")
    print(f"  Hermitian: {np.allclose(result.matrix, result.matrix.conj().T)}")
    print(f"  bundle files: {[path.name for path in outputs]}")

Round-trip and export
  transformed sites: 5
  matrix shape: (5, 5)
  Hermitian: True
  bundle files: ['matrix.npz', 'model.json', 'metadata.json', 'lattice.json', 'manifest.json']
